In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

In [ ]:
import torch
import torchvision
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import time
from pathlib import Path

# The standard device check — you'll use this pattern in every PyTorch notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version:     {torch.__version__}")
print(f"TorchVision version: {torchvision.__version__}")

# Tensors

## Tensor Question 1

In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
b = torch.zeros(2, 3)
c = torch.ones(4)

# Function to print tensor details
def print_tensor_details(name, tensor):
    print(f"Tensor {name}: ")
    print(f"Value:\n{tensor}")
    print(f"Shape:  {tensor.shape}")
    print(f"Dtype:  {tensor.dtype}")
    print(f"Device: {tensor.device}\n")
print_tensor_details("a",a)
print_tensor_details("b",b)
print_tensor_details("c",c)

# tensors on CPU right now
# Q: If you were running a training loop on the GPU, 
# why would it matter that your model weights and your input tensors are on the same device?
# A: It is important that all of them will be ran on the same device becase of memory usage. 
# GPU using not the same memory as CPU. So Pytorch will throw an runtime error.

## Tensor Question 2

Compute and print the element-wise square root using torch.sqrt().
Compute and print the sum using .sum().
Compute and print the mean using .mean().
Find and print the index of the maximum value using .argmax().

In [ ]:
x = torch.tensor([1.0, 4.0, 9.0, 16.0, 25.0])
squareroot = torch.sqrt(x)
print(f"squareroot of tensor x:\n {squareroot}") 
sum_x = torch.sum(x)
print(f"sum of tensor x: {sum_x}") 
mean_x = x.mean()
print(f"mean of tensor x: {mean_x}")
argmax_x = x.argmax()
print(f"argmax of tensor x: {argmax_x}")
# Q: Add a comment: .argmax() appears in nearly every inference example you'll encounter. 
# In the context of a classifier that outputs scores for 1,000 classes,
# what does .argmax() give you?
# A: .argmax() will return the index of the class with the highest score.




## Tensor Question 3

In [ ]:
# Moving to GPU 
a_gpu   = a.to(device)
print(f"a_gpu device: {a_gpu.device}")
# Moving back to CPU
a_back  = a_gpu.cpu()
print(f"a_gpu device: {a_back.device}")

a_numpy = a_back.numpy()
print(f"numpy type: {type(a_numpy)}")
print(f"numpy values:\n{a_numpy}")

# Q: Add a comment: why does PyTorch require .cpu() before you can call .numpy()? 
# A: because  NumPy is strictly a CPU-based library and has no concept of GPU memory addresses
# Q: What does this tell you about where NumPy arrays live?
# A: NumPy arrays exclusively live in System RAM (the CPU's memory).

## Tensor Question 4

In [ ]:
t = torch.arange(24).float()
print(t)
# 1) Reshape t to (4, 6) and print the shape
t_4_6 = t.reshape(4,6)
print(t_4_6.shape)
print(t_4_6)
print("--------------")

# 2) Reshape t to (2, 3, 4) and print the shape
t_2_3_4 = t.reshape(2,3,4)
print(t_2_3_4.shape)
print(t_2_3_4)
print("--------------")

# 3) Take result from step 1 and add a new dimension at position 0
t_with_batch =  t_4_6.unsqueeze(0)
print("Shape after unsqueeze(0):", t_with_batch.shape)
print(t_with_batch)

# A single image tensor is usually (channels, height, width).
# Neural networks expect input as (batch_size, channels, height, width).
# We use unsqueeze(0) to add a batch dimension for one image (batch_size=1).
# This matters because model layers are built to process batched tensors, so
# without this extra dimension the input shape will not match what the network expects.

## Tensor Question 5

In [ ]:
# Compare PyTorch and NumPy for matrix multiplication. 
# Starting from:
np_a = np.array([[1.0, 2.0], [3.0, 4.0]])
np_b = np.array([[5.0, 6.0], [7.0, 8.0]])

t_a  = torch.tensor(np_a, dtype=torch.float32)
t_b  = torch.tensor(np_b, dtype=torch.float32)

np_prod = np_a @ np_b
print(f"numPy result is: {np_prod}")

pt_prod = t_a @ t_b
print(f"pyTorch result is: {np_prod}")

# Confirm outputs match
matches = np.allclose(np_prod, pt_prod.numpy())
print("Do results match?", matches)

# Matrix multiplication combines input features
# with learned weights to produce new feature values (the linear transform)
# before applying bias and non-linear activation.

# Pretrained Models

Training a CNN from scratch requires millions of labeled images and days of GPU time. Pretrained models skip all of that — TorchVision ships with CNNs that were already trained on ImageNet, a dataset of over one million images across 1,000 categories. These models have learned to recognize edges, textures, shapes, and high-level objects. Loading one takes two lines of code. In practice, pretrained models are the default starting point for almost every real-world computer vision project.

## Model Question 1

In [ ]:
weights = ResNet18_Weights.DEFAULT
model   = models.resnet18(weights=weights)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# ResNet18 has roughly 11 million parameters.
# Training it from scratch required approximately 1.2 million
# labeled ImageNet images and days of multi-GPU compute. 
# What does that tell you about the practical value of starting from pretrained
# weights when you're on a deadline or a budget?
#  A: Save training time and compute resources
#     Achieve strong performance with minimal setup
#     Apply deep learning models to real-world problems quickly



## Model Question 2

In [ ]:
print(model)
# Q: What is the name of the final layer in ResNet18, and what is its output size? 
# (This number is the total count of ImageNet categories the model can predict.)
# A: The final layer in ResNet18 is fc (Linear(in_features=512, out_features=1000, bias=True))
#    So its output size is 1000, meaning it predicts scores for 1000 ImageNet classes.
# Q: Can you identify the blocks named layer1 through layer4? These are the "deep" part of the network — the feature extractor.
# A: Each of the layers include 2 Basic blocks
# Q: In plain terms, what does it mean for a network to be "deep"?
# A: “deep” means the network has many stacked layers/operations.
 

## Model Question 3

In [ ]:
# Before running inference, move the model to the GPU and set it to evaluation mode
model = model.to(device)
model.eval()
print("Model ready for inference.")
# Q: What does .to(device) do,
# and why does it need to match the device your input tensors will be on?
# A: .to(device) moves a model or tensor to a target device (like CPU or GPU), and can also change dtype if specified.
#    For a model, it moves all parameters and buffers; for tensors, it moves that tensor’s storage.
# Q: What does model.eval() change about the model's behavior? Name at least one layer type that behaves 
#    differently in training mode vs. evaluation mode.
# A: model.eval() changes the model behaviour from traning to inference and changes behaviour of certain layers from training
#    to predicting only. Layers like dropout and batch normalization

## Model Question 4

In [ ]:
preprocess = weights.transforms()
print(preprocess)

# This prints the full transform chain.
# Add a comment describing in plain English what each step does and why it matters.
# Address:
# What does the resize/crop step accomplish?
# this step prepare the image to be "digestable" for the model. 
# What does ToTensor() do to the pixel value range?
# ToTensor() takes pixel values that are usually 0 to 255 and rescales them to 0.0 to 1.0.
# What is normalization doing, and why does it use ImageNet's specific mean and standard deviation values rather than,
# say, mean=0.5, std=0.5?
# Normalization is “re-centering” and “re-scaling” pixel channels so the model sees inputs in the range/style it was trained on.
# it use ImageNet's specific mean and standard deviation values because the model was pretrained on those values

# Running Inference

With a model loaded and preprocessing defined, running inference on a new image takes about five lines of code. This section walks you through each step using real images from the Intel Image Classification dataset.
It picks a random image from a given scene class:

In [ ]:
import random
import torch.nn.functional as F

random.seed(42)

DATA_DIR = Path('/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test')
LABELS   = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

def load_sample_image(label):
    """Load a random image file from the given class folder."""
    class_dir = DATA_DIR / label
    img_path  = random.choice(list(class_dir.glob("*.jpg")))
    return Image.open(img_path).convert("RGB"), img_path.name

# Get the ImageNet class labels from the weights metadata — no separate download needed:

imagenet_classes = weights.meta["categories"]
print(f"Number of classes: {len(imagenet_classes)}")
print(f"First 5 labels: {imagenet_classes[:5]}")

## Inference Question 1

In [ ]:
# Write a function that runs inference on a single PIL image and returns the top-5 predicted class names and their probabilities.
# The function signature and steps are given below — fill in the implementation:

def get_top5_predictions(model, preprocess, image, device, class_labels):
    """
    Run inference on a PIL image and return the top-5 predictions.
    Returns a list of (class_name, probability) tuples.
    """
    model.eval()
    x = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)                    
        probs = F.softmax(logits, dim=1)
    top_probs, top_indices = torch.topk(probs, k=5, dim=1)
    top_probs = top_probs[0].cpu().tolist()
    top_indices = top_indices[0].cpu().tolist()
    return [(class_labels[i], p) for i, p in zip(top_indices, top_probs)]

img, img_name = load_sample_image("mountain")
preds         = get_top5_predictions(model, preprocess, img, device, imagenet_classes)

print(f"\nTop-5 predictions for '{img_name}':")
for class_name, prob in preds:
    print(f"  {class_name:30s}  {prob:.4f}")

# Q: does the top prediction make sense? 
# Remember that the model was trained on ImageNet's 1,000 categories,
# which include things like "alp", "valley", and "lakeside" rather than simply "mountain".
# Do any of the top-5 labels map onto what you'd describe as a mountain scene?
# A: Top predictions does make a lot of sense: "alp" is the top class, and labels like "valley" and "promontory" also match mountain terrain. 
# Yes : alp, volcano, valley,promontory they all describe mountain scene

## Inference Question 2

In [ ]:
LABELS = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

for label in LABELS:
    img, img_name = load_sample_image(label)
    preds = get_top5_predictions(model, preprocess, img, device, imagenet_classes)[:5]
    print(f"\n[{label}]  {img_name}")
    for class_name, prob in preds:
        print(f"  {class_name:30s}  {prob:.4f}")
# Add a comment: which classes does the model seem most confident about (high top-1 probability)? 
# A: looks like the highest confidence is on a mountain prediction : 
# ski                             0.5933
# alp                             0.3821
# Which does it seem least confident about?
# A: Least confident is on a street:
# unicycle                        0.1299
# triumphal arch                  0.1268
# Is there a pattern?
# The model seems most confident on mountain-like scenes (e.g., "ski"/"alp") and reasonably confident on some buildings/glacier images,
# but least confident on street and sea where top-1 probabilities are low and spread across multiple labels.

## Inference Question 3

In [ ]:
img, image_name = load_sample_image("forest")
input_tensor = preprocess(img).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_tensor)

probs = torch.nn.functional.softmax(logits[0], dim=0)

print(f"Logit  range: min={logits.min():.2f}, max={logits.max():.2f}")
print(f"Prob   range: min={probs.min():.6f}, max={probs.max():.4f}")
print(f"Probs sum to: {probs.sum():.6f}")
print(f"Top prediction: {imagenet_classes[probs.argmax().item()]}  ({probs.max():.4f})")
 # why do neural networks output logits internally rather than probabilities? 
# NN output logits internally because of the numerical stability and also operations for floating points rounding "vanishing" gradient.
#  Logits preserve better distances betqween class scores more accurately
# In a production pipeline that needs to filter out low-confidence predictions,
# which representation would you work with — logits or probabilities — and why?
# Probabilities will work better for less advancedcomputations in the programm . It's the standard way to say "I'm sure" or "I'm guessing."
# Logits will work only for training the model or doing very advanced math where needed the "unfiltered" truth.

## Inference Question 4

In [ ]:
# Create a visualization that shows an image alongside a horizontal bar chart of its top-5 predictions.
# Use plt.subplots(1, 2) with one panel for the image and one for the bar chart. 
# Save it to outputs/warmup_inference_viz.png. 
# You have all the pieces — img, preds, and plt — from the previous questions.
# Q: how would you adapt this kind of visualization for a dashboard
# that a non-technical team member needs to review flagged predictions? 
# A: For a non-technical dashboard, I would keep it simple and decision ready: colors for confident description: red, yellow and green
# for the most confident predictions. also I would show only top confident decisions for easier scanning.

# Q: What threshold on the top-1 probability might you use to decide when a prediction is "confident enough" to act on?
# A: Confident enough to act: top-1 probability >= 0.70 (70%+)
# Needs review: 0.40 to 0.69
# Low confidence: < 0.40
# If errors are costfull make treshold even higher let say 0.80

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].imshow(img)
ax[0].set_title(image_name)
ax[0].axis("off")

class_names = [name for name, _ in preds][::-1]
probs = [p for _, p in preds][::-1]

ax[1].barh(class_names, probs, color="steelblue")
ax[1].set_title("Top-5 predictions")
ax[1].set_xlabel("Probability")
ax[1].set_xlim(0, 1)

for i, p in enumerate(probs):
    ax[1].text(p + 0.01, i, f"{p:.3f}", va="center")

plt.tight_layout()
plt.savefig("outputs/warmup_inference_viz.png", dpi=150, bbox_inches="tight")
plt.show()

# For a non-technical dashboard: show image + top prediction + confidence color (green/yellow/red),
# and auto-flag low-confidence cases for review. A reasonable starting threshold is top-1 >= 0.70
# as "confident enough"; below that goes to human review.
